# OLYMPUS TECH 4.0  Part Two Data Cleaning


In [ ]:
!pip -q install -U scikit-learn seaborn

In [ ]:
# Setup, imports, and data configuration
import os
import re
import json
import time
import zipfile
import warnings
from collections import Counter, OrderedDict

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import RepeatedKFold
from sklearn.metrics import r2_score, mean_squared_error

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

TEAM_NAME = "The gentlemen"
RANDOM_STATE = 42
INPUT_CSV_PATH = "train.csv"
TEST_CSV_PATH = "test.csv"  
V2_CLEANED_PATH = "v2_cleaned_dataset.csv" 

OUTPUT_CLEANED_CSV_PATH = f"{TEAM_NAME}_cleaned_dataset.csv"
OUTPUT_CLEANED_TEST_PATH = f"{TEAM_NAME}_cleaned_test_dataset.csv"
OUTPUT_AUDIT_JSON_PATH = "cleaning_audit_report.json"
OUTPUT_MAPPING_CSV_PATH = "categorical_mapping_table.csv"
PLOT_DIR = "cleaning_diagnostic_plots"
ZIP_FILENAME = f"{TEAM_NAME}_Part2_Cleaning_Deliverables.zip"

CV_SPLITS, CV_REPEATS = 5, 3  
APPEND_QC_COLUMNS = True  

os.makedirs(PLOT_DIR, exist_ok=True)

if not os.path.exists(INPUT_CSV_PATH):
    try:  
        from google.colab import files as _colab_files

        print("Upload train.csv (and optionally test.csv / v2_cleaned_dataset.csv)")
        _colab_files.upload()
    except ImportError:
        raise FileNotFoundError(f"'{INPUT_CSV_PATH}' not found in {os.getcwd()}")

df_raw = pd.read_csv(INPUT_CSV_PATH)
ORIGINAL_COLUMNS = list(df_raw.columns)
TARGET = "settlement_index"
print(f"Loaded '{INPUT_CSV_PATH}' -> {df_raw.shape[0]} rows x {df_raw.shape[1]} columns")
assert df_raw.shape[1] == 31 and TARGET in df_raw.columns

In [ ]:
# Categorical taxonomy normalization


def _norm(val):
    if val is None or (isinstance(val, float) and np.isnan(val)) or pd.isna(val):
        return None
    s = re.sub(r"\s+", " ", str(val).strip().lower())
    return s or None


STAR_MAP = {
    "m-type red dwarf": "M-type red dwarf", "red dwarf": "M-type red dwarf",
    "m-class red dwarf": "M-type red dwarf", "red dwarf (m-type)": "M-type red dwarf",
    "m-type main sequence": "M-type red dwarf",
    "g-type main sequence": "G-type main sequence",
    "k-type main sequence": "K-type main sequence",
    "k-type orange dwarf": "K-type orange dwarf", "orange dwarf": "K-type orange dwarf",
    "f-type main sequence": "F-type main sequence",
    "a-type main sequence": "A-type main sequence", "blue supergiant": "A-type main sequence",
    "blue giant binary": "A-type main sequence",
    "close binary (tidal heating)": "Close binary (tidal heating)",
    "close binary (k-type + m-type)": "Close binary (tidal heating)",
    "red dwarf (close binary)": "Close binary (tidal heating)",
    "binary system (g+m)": "Binary system (G+M)", "binary system (g-type + m-type)": "Binary system (G+M)",
    "g-type main sequence + m-type red dwarf binary": "Binary system (G+M)",
    "g-type main sequence + m-type dwarf binary": "Binary system (G+M)",
    "m-type red dwarf + g-type main sequence": "Binary system (G+M)",
    "red dwarf + g-type main sequence binary": "Binary system (G+M)",
    "binary system (k+k)": "Binary system (K+K)", "k-type main sequence binary": "Binary system (K+K)",
    "red dwarf + k-type binary": "Binary system (K+K)", "red dwarf binary (k-type + k-type)": "Binary system (K+K)",
    "m-type red dwarf (binary k+k system)": "Binary system (K+K)",
    "binary system (f+g)": "Binary system (F+G)", "binary system (f-type + g-type main sequence)": "Binary system (F+G)",
    "f-type main sequence + g-type main sequence binary": "Binary system (F+G)",
    "g-type main sequence + f-type main sequence binary": "Binary system (F+G)",
    "wide binary": "Wide binary", "red dwarf + binary companion": "Wide binary",
    "subdwarf": "Subdwarf",
    "triple star system": "Triple star system", "red dwarf triple system": "Triple star system",
    "orange giant": "Orange giant", "red giant": "Red giant",
    "white dwarf remnant": "White dwarf remnant", "white dwarf": "White dwarf remnant",
    "brown dwarf companion": "Brown dwarf companion", "brown dwarf": "Brown dwarf companion",
    "red dwarf with brown dwarf companion": "Brown dwarf companion",
}


def clean_star_type(val):
    s = _norm(val)
    if s is None:
        return np.nan
    if s in STAR_MAP:
        return STAR_MAP[s]
    # Fallback for unseen variants
    if "binary" in s or "+" in s:
        if "g" in s and "m" in s:
            return "Binary system (G+M)"
        if "f" in s and "g" in s:
            return "Binary system (F+G)"
        return "Binary system (K+K)"
    for key, out in [("white dwarf", "White dwarf remnant"), ("brown dwarf", "Brown dwarf companion"),
                     ("red giant", "Red giant"), ("orange giant", "Orange giant"),
                     ("red dwarf", "M-type red dwarf"), ("orange dwarf", "K-type orange dwarf")]:
        if key in s:
            return out
    return str(val).strip()


def clean_planet_type(val):
    s = _norm(val)
    if s is None:
        return np.nan, 0
    tl = int("tidally locked" in s)
    rest = s.replace("tidally locked", "").strip()
    head = re.split(r"\s+with\s+|\+", rest)[0].strip()  
    if "rogue" in s:
        return "rogue planet", tl
    if rest == "":
        return "tidally locked", 1 
    if head in ("relic world",):
        return "relic world", tl  
    if "relic" in head or "artificial" in head:
        return "relic/artificial world", tl
    if "super-earth" in head:
        return "super-Earth", tl
    if "moon" in head:
        return "gas giant moon", tl
    if "frozen" in head:
        return "ice world", tl
    if "ocean" in head or "methane-sea" in head:
        return "ocean world", tl
    if "desert" in head:
        return "desert world", tl
    if "volcanic" in head and "cryo" not in head and not head.startswith("ice"):
        return "volcanic world", tl
    if "ice" in head or "cryo" in head:
        return "ice world", tl
    if "terrestrial" in head or "crystalline" in head:
        return "terrestrial", tl
    return head, tl


WATER_TOKENS = {"global ocean": "global ocean", "seas": "seas", "lakes": "seas", "liquid pools": "seas",
                "subsurface": "subsurface", "subsurface ocean": "subsurface", "subsurface ice": "subsurface",
                "trace": "trace", "vapor": "trace", "ice": "ice", "none": "none"}


def clean_water_presence(val):
    s = _norm(val)
    if s is None:
        return np.nan
    first = s.split("+")[0].strip()
    if first in WATER_TOKENS:
        return WATER_TOKENS[first]
    for key, out in [("ocean", "global ocean"), ("sea", "seas"), ("subsurface", "subsurface"),
                     ("trace", "trace"), ("ice", "ice"), ("none", "none")]:
        if key in first:
            return out
    return first


SURFACE_CANON = ["rocky", "volcanic", "ice", "crystalline", "sandy", "metallic"]


def _surface_token(tok):
    tok = tok.strip()
    if not tok:
        return None
    if "basalt" in tok: 
        return "rocky"
    if "volcanic" in tok or "ash" in tok:
        return "volcanic"
    if "ice" in tok or "icy" in tok or "frozen" in tok:
        return "ice"
    if "crystal" in tok:
        return "crystalline"
    if "sand" in tok:
        return "sandy"
    if "metal" in tok:
        return "metallic"
    if "rock" in tok or "seafloor" in tok or "forest" in tok:
        return "rocky"
    return None


def clean_surface_composition(val):
    s = _norm(val)
    if s is None:
        return np.nan, "none"
    s = s.replace("crystalline ice", "ice+crystalline").replace("water ice", "ice")
    s = s.replace("crystalline methane", "crystalline")
    parts = re.split(r"\+|\s+with\s+|\s+and\s+", s)
    toks = []
    for p in parts:
        p = p.strip()
        # Compound surface tokens
        sub = [p] if p in ("volcanic rock", "rocky seafloor", "silicate sand", "volcanic ash") else p.split(" ")
        if len(sub) > 1 and all(_surface_token(x) for x in sub if x not in ("mineral", "deposits")):
            cand = [_surface_token(x) for x in sub if x not in ("mineral", "deposits")]
        else:
            cand = [_surface_token(p)]
        for c in cand:
            if c and c not in toks:
                toks.append(c)
    if not toks:
        return s, "none"
    return toks[0], (toks[1] if len(toks) > 1 else "none")


def clean_primary_hazard(val):
    s = _norm(val)
    if s is None:
        return np.nan
    s = s.replace(" ", "_")
    if s == "none":
        return "none"
    if "thermal_vent" in s:
        return "thermal_vents"
    if "seasonal" in s and "storm" in s:
        return "seasonal_storms"
    if any(k in s for k in ("cold", "heat", "temperature")):
        return "temperature_extremes"
    lead = s.split("+")[0]
    for keys, out in [(("seismic", "volcanic", "quake"), "seismic"), (("storm", "wind"), "storms"),
                      (("radiation", "electromagnetic", "flare"), "radiation"),
                      (("toxic", "microbial", "pressure", "acid"), "toxic_air"),
                      (("predator",), "predators"), (("unknown", "anomaly"), "unknown_anomaly")]:
        if any(k in lead for k in keys):
            return out
    return "unknown_anomaly"


def clean_rings(val):
    s = _norm(val)
    if s is None:
        return np.nan
    if s in ("yes", "true", "1", "y", "1.0"):
        return "yes"
    if s in ("no", "false", "0", "n", "0.0"):
        return "no"
    return np.nan


ORDINAL_LEVELS = {  
    "storm_frequency": ["rare", "seasonal", "frequent", "constant"],
    "seismic_activity": ["low", "moderate", "high", "extreme"],
    "magnetic_field": ["weak", "moderate", "strong"],
    "radiation_level": ["low", "moderate", "high", "extreme"],
    "energy_harvest_potential": ["low", "moderate", "high", "extreme"],
}

In [ ]:
# Atmosphere composition parser
GAS_CANON = {"n2": "N2", "o2": "O2", "co2": "CO2", "ch4": "CH4", "ar": "Ar", "h2": "H2", "he": "He",
             "so2": "SO2", "h2o": "H2O", "c2h6": "C2H6", "ne": "Ne", "nh3": "NH3", "h2s": "H2S", "co": "CO",
             "o3": "O3", "kr": "Kr", "xe": "Xe"}
TRACE_ALIASES = {"trace", "trace gases", "tr", "other", "others", "other gases", "trace gas"}
# symbol, optional 'gases', separator (space or ':' / '='), number, optional '%'
GAS_RE = re.compile(r"([A-Za-z][A-Za-z0-9]*(?:\s+gases?)?)(?:\s*[:=]\s*|\s+)(\d+(?:\.\d+)?)\s*%?")


def parse_gases(s):
    info = {"unknown_tokens": [], "bare_trace": False}
    if not isinstance(s, str) or not s.strip():
        return OrderedDict(), info
    txt = re.sub(r"[,+;/|]", " ", s)
    out = OrderedDict()
    for sym, val in GAS_RE.findall(txt):
        key = sym.strip().lower()
        v = float(val)
        if key in TRACE_ALIASES:
            out["Trace"] = out.get("Trace", 0.0) + v
        elif key in GAS_CANON:
            g = GAS_CANON[key]
            out[g] = out.get(g, 0.0) + v
        else:
            info["unknown_tokens"].append(sym)
    # unquantified trace token, e.g. 'N2 78% O2 18% Ar 3% trace'
    stripped = GAS_RE.sub(" ", txt)
    if re.search(r"\b(trace|tr|other)\b", stripped, re.IGNORECASE) and "Trace" not in out:
        info["bare_trace"] = True
        rem = round(100.0 - sum(out.values()), 4)
        out["Trace"] = rem if 0 < rem <= 5 else 0.0
    return out, info


def gas_string(d):
    if not d:
        return "Unknown"
    items = sorted([(k, v) for k, v in d.items() if k != "Trace"], key=lambda kv: (-kv[1], kv[0]))
    if d.get("Trace", 0) > 0:
        items.append(("Trace", d["Trace"]))
    return " ".join(f"{k} {int(round(v))}%" if abs(v - round(v)) < 1e-9 else f"{k} {round(v, 2)}%" for k, v in items)


GAS_COLUMNS = {"oxygen_percent": "O2", "co2_percent": "CO2", "methane_percent": "CH4"}

In [ ]:
# Physical and consistency repairs
ABS_ZERO_C = -273.15
MEAN_TEMP_PLACEHOLDER = 1.0
ROTATION_TO_ORBIT_MAX = 3.0 


def n_decimals(x):
    if pd.isna(x):
        return np.nan
    r = repr(round(float(x), 9)) 
    if "e" in r:
        return 16
    return len(r.split(".")[1].rstrip("0")) if "." in r else 0


def stateless_clean(df_in):
    df = df_in.copy()
    log = Counter()
    qc = pd.DataFrame(index=df.index)

    # Identifiers
    base = df["planet_name"].astype(str).str.strip().str.replace(r"_\d+$", "", regex=True)
    log["planet_name_suffix_restored"] = int((~df["planet_name"].astype(str).str.contains(r"_\d+$")).sum())
    df["planet_name"] = base + "_" + df["planetId"].astype(int).astype(str)

    # Categoricals
    df["star_type"] = df["star_type"].map(clean_star_type)
    pt = df["planet_type"].map(clean_planet_type)
    df["planet_type"] = pt.map(lambda t: t[0])
    qc["qc_tidally_locked"] = pt.map(lambda t: t[1]).astype(int)
    df["water_presence"] = df["water_presence"].map(clean_water_presence)
    sc = df["surface_composition"].map(clean_surface_composition)
    df["surface_composition"] = sc.map(lambda t: t[0])
    qc["qc_surface_secondary"] = sc.map(lambda t: t[1])
    df["primary_hazard"] = df["primary_hazard"].map(clean_primary_hazard)
    df["rings"] = df["rings"].map(clean_rings)
    for c, levels in ORDINAL_LEVELS.items():
        v = df[c].map(_norm)
        bad = v.notna() & ~v.isin(levels)
        log[f"{c}_invalid_tokens"] = int(bad.sum())
        df[c] = v.where(~bad, np.nan)

    rogue = df["planet_type"].eq("rogue planet")
    tl = qc["qc_tidally_locked"].eq(1) | df["planet_type"].eq("tidally locked")

    # Disguised missing value
    ph = df["mean_temp_c"].eq(MEAN_TEMP_PLACEHOLDER)
    qc["qc_mean_temp_placeholder"] = ph.astype(int)
    log["mean_temp_placeholder_1.0_to_nan"] = int(ph.sum())
    df.loc[ph, "mean_temp_c"] = np.nan

    # Thermodynamics
    imp_range = df["mean_temp_c"].notna() & (df["mean_temp_c"] - df["temp_range_c"] / 2 < ABS_ZERO_C)
    qc["qc_temp_range_below_abs_zero"] = imp_range.astype(int)
    log["temp_range_below_absolute_zero_to_nan"] = int(imp_range.sum())
    df.loc[imp_range, "temp_range_c"] = np.nan
    df.loc[df["mean_temp_c"] < ABS_ZERO_C, "mean_temp_c"] = np.nan

    # Orbital mechanics
    conflict = rogue & (df["orbital_distance_au"].notna() | df["orbital_period_days"].notna())
    qc["qc_rogue_orbit_conflict"] = conflict.astype(int)
    log["rogue_planets_with_bound_orbit"] = int(conflict.sum())
    log["rogue_planets_orbit_not_applicable"] = int(rogue.sum())
    df.loc[rogue, ["orbital_distance_au", "orbital_period_days"]] = np.nan  # set to 0 after imputation

    # Synchronous rotation for tidally locked worlds
    unit = tl & df["day_length_hours"].notna() & (
        (df["day_length_hours"] - df["orbital_period_days"]).abs() < 1e-9)
    qc["qc_day_length_unit_fixed"] = unit.astype(int)
    log["tidally_locked_day_length_days_to_hours"] = int(unit.sum())
    df.loc[unit, "day_length_hours"] = df.loc[unit, "orbital_period_days"] * 24.0

    ratio = df["day_length_hours"] / (df["orbital_period_days"] * 24.0)
    bad_rot = ~rogue & (ratio > ROTATION_TO_ORBIT_MAX)
    qc["qc_rotation_exceeds_orbit"] = bad_rot.astype(int)
    log["rotation_period_gt_3x_orbit_to_nan"] = int(bad_rot.sum())
    df.loc[bad_rot, "day_length_hours"] = np.nan

    # Hard physical domain checks (none fire on this archive, kept for test.csv)
    for c, lo, hi in [("albedo", 0, 1), ("cloud_coverage_percent", 0, 100), ("rare_mineral_index", 0, 100),
                      ("axial_tilt_deg", 0, 180), ("oxygen_percent", 0, 100), ("co2_percent", 0, 100),
                      ("methane_percent", 0, 100), ("moons_count", 0, 200)]:
        bad = (df[c] < lo) | (df[c] > hi)
        log[f"{c}_out_of_domain_to_nan"] = int(bad.sum())
        df.loc[bad, c] = np.nan
    for c in ["orbital_distance_au", "orbital_period_days", "day_length_hours", "temp_range_c", "gravity_g",
              "escape_velocity_kms", "atmospheric_pressure_atm"]:
        bad = df[c] <= 0
        log[f"{c}_non_positive_to_nan"] = int(bad.sum())
        df.loc[bad, c] = np.nan

    # Conflicts flagged but not altered (insufficient evidence which field is wrong)
    liquid = df["water_presence"].isin(["seas", "global ocean"])
    qc["qc_liquid_water_temp_conflict"] = (liquid & ((df["mean_temp_c"] > 150) | (df["mean_temp_c"] < -100))).astype(int)
    r_imp = (df["escape_velocity_kms"] * 1e3) ** 2 / (2 * df["gravity_g"] * 9.81) / 6.371e6
    qc["qc_escape_gravity_inconsistent"] = ((r_imp > 20) | (r_imp < 0.05)).astype(int)
    log["flag_only_liquid_water_temperature_conflicts"] = int(qc["qc_liquid_water_temp_conflict"].sum())
    log["flag_only_escape_velocity_gravity_inconsistent"] = int(qc["qc_escape_gravity_inconsistent"].sum())

    return df, qc, dict(log)

In [ ]:
# Leakage-free fitted cleaner
LOG_COLS = ["orbital_distance_au", "orbital_period_days", "day_length_hours", "temp_range_c",
            "atmospheric_pressure_atm", "escape_velocity_kms", "gravity_g"]
IMPUTE_NUM = ["orbital_distance_au", "orbital_period_days", "day_length_hours", "axial_tilt_deg", "mean_temp_c",
              "temp_range_c", "gravity_g", "escape_velocity_kms", "atmospheric_pressure_atm", "albedo",
              "cloud_coverage_percent", "moons_count", "rare_mineral_index", "oxygen_percent", "co2_percent",
              "methane_percent"]
CAT_COLS = ["star_type", "planet_type", "water_presence", "surface_composition", "storm_frequency",
            "seismic_activity", "magnetic_field", "radiation_level", "rings", "energy_harvest_potential",
            "primary_hazard"]
HARD_BOUNDS = {"mean_temp_c": (ABS_ZERO_C, None), "albedo": (0, 1), "cloud_coverage_percent": (0, 100),
               "rare_mineral_index": (0, 100), "axial_tilt_deg": (0, 180), "moons_count": (0, None),
               "oxygen_percent": (0, 100), "co2_percent": (0, 100), "methane_percent": (0, 100)}


class PlanetaryCleaner:

    def __init__(self, random_state=RANDOM_STATE, imputer_rounds=6, fast=False):
        self.random_state = random_state
        self.imputer_rounds = imputer_rounds
        self.fast = fast

    # Gases
    def _gas_frame(self, df):
        parsed = df["atmosphere_primary_gases"].map(parse_gases)
        return parsed.map(lambda t: t[0]), parsed.map(lambda t: t[1])

    def _fit_gas(self, df, gases):
        sig = gases.map(lambda d: tuple(sorted(k for k in d if k != "Trace")))
        tot = gases.map(lambda d: sum(d.values()))
        ok = (tot - 100).abs() < 1.0
        self.gas_templates_ = {}
        for s, idx in sig[ok].groupby(sig[ok]).groups.items():
            if len(idx) >= 5:
                self.gas_templates_[s] = {g: float(np.median([gases[i][g] for i in idx])) for g in s}
        self.trace_median_, self.trace_cap_ = {}, {}
        for col, g in GAS_COLUMNS.items():
            absent = ~gases.map(lambda d: g in d) & df[col].notna()
            vals = df.loc[absent, col]
            self.trace_cap_[col] = float(vals.quantile(0.995)) if len(vals) else 1.0
            self.trace_median_[col] = vals.groupby(df.loc[absent, "planet_type"]).median().to_dict()
            self.trace_median_[col]["__all__"] = float(vals.median()) if len(vals) else 0.0

    def _repair_composition(self, d):
        tot = sum(d.values())
        if abs(tot - 100) < 5:
            return d, None
        s = tuple(sorted(k for k in d if k != "Trace"))
        tmpl = self.gas_templates_.get(s)
        if not tmpl:
            return d, None
        dev = {g: abs(d[g] - tmpl[g]) for g in s}
        worst = max(dev, key=dev.get)
        resid = 100.0 - (tot - d[worst])
        if resid <= 0 or abs(resid - tmpl[worst]) >= dev[worst]:
            return d, None  # repair would not move it towards the template: leave + flag
        d2 = OrderedDict(d)
        d2[worst] = round(resid, 4)
        return d2, (worst, d[worst], round(resid, 4))

    def _apply_gas(self, df, qc, log):
        gases, infos = self._gas_frame(df)
        repaired, rep_log = [], []
        for i, d in gases.items():
            # lone 'CO' with no CO2 while co2_percent equals it -> transcription typo for CO2
            if "CO" in d and "CO2" not in d and pd.notna(df.at[i, "co2_percent"]) and abs(df.at[i, "co2_percent"] - d["CO"]) < 0.51:
                d = OrderedDict((("CO2" if k == "CO" else k), v) for k, v in d.items())
                log["gas_CO_typo_to_CO2"] += 1
            d2, r = self._repair_composition(d)
            if r:
                rep_log.append({"planetId": int(df.at[i, "planetId"]), "gas": r[0], "from": r[1], "to": r[2]})
            repaired.append(d2)
        gases = pd.Series(repaired, index=df.index)
        tot = gases.map(lambda d: sum(d.values()))
        qc["qc_gas_sum_invalid"] = ((tot - 100).abs() >= 5).astype(int)
        log["gas_composition_template_repairs"] = len(rep_log)
        log["gas_sum_still_invalid_flagged"] = int(qc["qc_gas_sum_invalid"].sum())
        log["gas_bare_trace_tokens"] = int(infos.map(lambda x: x["bare_trace"]).sum())
        log["gas_unknown_tokens"] = int(infos.map(lambda x: len(x["unknown_tokens"])).sum())
        repaired_ids = {r["planetId"]: r for r in rep_log}

        for col, g in GAS_COLUMNS.items():
            sv = gases.map(lambda d: d.get(g, np.nan))
            present = sv.notna()
            qc[f"qc_missing_{col}"] = df[col].isna().astype(int)
            fill = df[col].isna() & present
            log[f"{col}_recovered_from_gas_string"] = int(fill.sum())
            df.loc[fill, col] = sv[fill]
            dis = df[col].notna() & present & ((df[col] - sv).abs() > np.maximum(1.0, 0.2 * sv))
            log[f"{col}_column_string_conflicts_fixed"] = int(dis.sum())
            df.loc[dis, col] = sv[dis]
            for pid, r in repaired_ids.items():
                if r["gas"] == g:
                    df.loc[df["planetId"] == pid, col] = r["to"]
            absent_nan = df[col].isna() & ~present
            med = self.trace_median_[col]
            df.loc[absent_nan, col] = df.loc[absent_nan, "planet_type"].map(lambda p: med.get(p, med["__all__"])).astype(float)
            log[f"{col}_absent_gas_trace_filled"] = int(absent_nan.sum())
            # observed trace readings are left untouched; only *filled* values are bounded
            df.loc[absent_nan, col] = df.loc[absent_nan, col].clip(0, self.trace_cap_[col])
        df["atmosphere_primary_gases"] = gases.map(gas_string)
        return rep_log

    # Fit / transform
    def fit(self, df_train_raw):
        df, qc, _ = stateless_clean(df_train_raw)
        gases, _ = self._gas_frame(df)
        self._fit_gas(df, gases)
        # categorical fills: mode within planet_type, global fallback
        self.cat_modes_ = {}
        for c in CAT_COLS:
            glob = df[c].mode(dropna=True)
            self.cat_modes_[c] = {"__all__": glob.iloc[0] if len(glob) else "unknown",
                                  **df.groupby("planet_type")[c].agg(lambda x: x.mode().iloc[0] if x.notna().any() else np.nan).dropna().to_dict()}
        self.censor_caps_ = {}
        for c in IMPUTE_NUM:
            s = df[c].dropna()
            hp = s[s.map(n_decimals) > 6]
            vc = hp.value_counts()
            if len(vc) and vc.iloc[0] >= 10 and np.isclose(vc.index[0], hp.max()):
                self.censor_caps_[c] = float(vc.index[0])
        df_g = df.copy()
        self._apply_gas(df_g, qc.copy(), Counter())
        self._fill_cats(df_g)
        self.bounds_ = {}
        for c in IMPUTE_NUM:
            lo, hi = df_g[c].min(), df_g[c].max()
            hlo, hhi = HARD_BOUNDS.get(c, (None, None))
            lo = max(lo, hlo) if hlo is not None else lo
            hi = min(hi, hhi) if hhi is not None else hi
            self.bounds_[c] = (float(lo), float(hi))
        M = self._design(df_g, fit=True)
        est = HistGradientBoostingRegressor(max_iter=60 if self.fast else 120, learning_rate=0.1,
                                            max_leaf_nodes=31, random_state=self.random_state)
        self.imputer_ = IterativeImputer(estimator=est, max_iter=self.imputer_rounds, initial_strategy="median",
                                         skip_complete=True, random_state=self.random_state, tol=1e-3)
        self.imputer_.fit(M)
        return self

    def _fill_cats(self, df):
        filled = {}
        for c in CAT_COLS:
            m = df[c].isna()
            filled[c] = int(m.sum())
            if m.any():
                modes = self.cat_modes_[c]
                df.loc[m, c] = df.loc[m, "planet_type"].map(lambda p: modes.get(p, modes["__all__"])).fillna(modes["__all__"])
        return filled

    def _design(self, df, fit=False):
        X = df[IMPUTE_NUM].astype(float).copy()
        for c in LOG_COLS:
            X[c] = np.log(X[c])
        if fit:
            self.cat_levels_ = {c: sorted(df[c].astype(str).unique()) for c in CAT_COLS}
        for c in CAT_COLS:
            lv = {v: i for i, v in enumerate(self.cat_levels_[c])}
            X[f"_cat_{c}"] = df[c].astype(str).map(lv).fillna(-1).astype(float)
        X["_rogue"] = df["planet_type"].eq("rogue planet").astype(float)
        return X

    def transform(self, df_raw_in, return_log=False):
        df, qc, log = stateless_clean(df_raw_in)
        log = Counter(log)
        qc["qc_missing_orbit"] = (df[["orbital_distance_au", "orbital_period_days"]].isna().any(axis=1)
                                  & ~df["planet_type"].eq("rogue planet")).astype(int)
        for c in ["day_length_hours", "axial_tilt_deg", "escape_velocity_kms", "moons_count", "temp_range_c"]:
            qc[f"qc_missing_{c}"] = df[c].isna().astype(int)
        rep_log = self._apply_gas(df, qc, log)
        log["categorical_values_imputed"] = self._fill_cats(df)
        for c, cap in self.censor_caps_.items():
            qc[f"qc_censored_{c}"] = np.isclose(df[c], cap).astype(int)
            log[f"right_censored_{c}"] = int(qc[f"qc_censored_{c}"].sum())

        miss_before = df[IMPUTE_NUM].isna()
        rogue = df["planet_type"].eq("rogue planet")
        M = self._design(df)
        imp = pd.DataFrame(self.imputer_.transform(M), columns=M.columns, index=df.index)
        for c in IMPUTE_NUM:
            v = np.exp(imp[c]) if c in LOG_COLS else imp[c]
            lo, hi = self.bounds_[c]
            df.loc[miss_before[c], c] = v[miss_before[c]].clip(lo, hi)
        # coupled physical constraints on imputed values
        tmax = 2 * (df["mean_temp_c"] - ABS_ZERO_C)
        fix_r = miss_before["temp_range_c"] & (df["temp_range_c"] > tmax)
        df.loc[fix_r, "temp_range_c"] = tmax[fix_r]
        df["moons_count"] = df["moons_count"].round().astype(int)
        df.loc[rogue, ["orbital_distance_au", "orbital_period_days"]] = 0.0  # unbound: not applicable
        log["numeric_values_imputed"] = {c: int(miss_before[c].sum() - (rogue.sum() if "orbital" in c else 0)) for c in IMPUTE_NUM}

        out = df[ORIGINAL_COLUMNS] if TARGET in df.columns else df[[c for c in ORIGINAL_COLUMNS if c != TARGET]]
        if APPEND_QC_COLUMNS:
            out = pd.concat([out, qc], axis=1)
        self.last_imputed_mask_ = miss_before
        self.last_gas_repairs_ = rep_log
        return (out, dict(log)) if return_log else out

In [ ]:
# Fit cleaner and export cleaned dataset
t0 = time.time()
cleaner = PlanetaryCleaner().fit(df_raw)
df_cleaned, clean_log = cleaner.transform(df_raw, return_log=True)
imputed_mask = cleaner.last_imputed_mask_
print(f"Cleaner fitted + applied in {time.time() - t0:.1f}s -> {df_cleaned.shape}")

core = df_cleaned[ORIGINAL_COLUMNS]
assert list(core.columns) == ORIGINAL_COLUMNS, "Column schema mismatch"
assert len(df_cleaned) == len(df_raw), "Row count mismatch"
assert core.isnull().sum().sum() == 0, "Null values remain"
assert (core["mean_temp_c"] - core["temp_range_c"] / 2 >= ABS_ZERO_C - 1e-6).all(), "Absolute-zero violation"
assert core["albedo"].between(0, 1).all() and core["cloud_coverage_percent"].between(0, 100).all()
assert (core["settlement_index"] == df_raw["settlement_index"]).all(), "Target must never be modified"
df_cleaned.to_csv(OUTPUT_CLEANED_CSV_PATH, index=False)
print(f"Saved {OUTPUT_CLEANED_CSV_PATH}")

test_written = False
if os.path.exists(TEST_CSV_PATH):
    df_test_raw = pd.read_csv(TEST_CSV_PATH)
    df_test_clean = cleaner.transform(df_test_raw)  # train-fitted state only: no test leakage into train
    df_test_clean.to_csv(OUTPUT_CLEANED_TEST_PATH, index=False)
    test_written = True
    print(f"Saved {OUTPUT_CLEANED_TEST_PATH} ({len(df_test_clean)} rows)")

# categorical mapping table (raw label -> clean label, with diagnostic target statistics)
map_rows = []
for col in ["star_type", "planet_type", "water_presence", "surface_composition", "primary_hazard", "rings"]:
    tmp = pd.DataFrame({"raw": df_raw[col].astype(object).where(df_raw[col].notna(), "<NaN>"),
                        "clean": df_cleaned[col], TARGET: df_raw[TARGET]})
    g = tmp.groupby(["raw", "clean"])[TARGET].agg(["count", "mean"]).reset_index()
    g.insert(0, "column", col)
    map_rows.append(g)
mapping_table = pd.concat(map_rows).rename(columns={"count": "n_rows", "mean": "mean_settlement_index"})
mapping_table.to_csv(OUTPUT_MAPPING_CSV_PATH, index=False)

In [ ]:
# Repeated 5-fold cross-validation benchmark


def make_model():
    return HistGradientBoostingRegressor(max_iter=500, learning_rate=0.04, max_leaf_nodes=31,
                                         min_samples_leaf=20, l2_regularization=1.0,
                                         categorical_features="from_dtype", random_state=RANDOM_STATE)


def to_matrix(X, cat_levels=None):
    X = X.drop(columns=[c for c in ["planetId", "planet_name", TARGET, "atmosphere_primary_gases"] if c in X.columns]).copy()
    obj = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    cat_levels = cat_levels or {c: sorted(X[c].dropna().astype(str).unique()) for c in obj}
    for c in obj:
        X[c] = pd.Categorical(X[c].astype(object).where(X[c].notna(), None), categories=cat_levels[c])
    return X, cat_levels


def evaluate(name, fold_fn, splits, y):
    r2s, rmses = [], []
    for tr, va in splits:
        Xtr, Xva = fold_fn(tr, va)
        m = make_model().fit(Xtr, y.iloc[tr])
        p = m.predict(Xva)
        r2s.append(r2_score(y.iloc[va], p))
        rmses.append(float(np.sqrt(mean_squared_error(y.iloc[va], p))))
    print(f"  {name:<34} R² {np.mean(r2s):.4f} ± {np.std(r2s):.4f} | RMSE {np.mean(rmses):.4f} ± {np.std(rmses):.4f}")
    return {"r2_mean": float(np.mean(r2s)), "r2_std": float(np.std(r2s)), "rmse_mean": float(np.mean(rmses)),
            "rmse_std": float(np.std(rmses)), "r2_folds": [float(v) for v in r2s], "rmse_folds": [float(v) for v in rmses]}


y_all = df_raw[TARGET].reset_index(drop=True)
splits = list(RepeatedKFold(n_splits=CV_SPLITS, n_repeats=CV_REPEATS, random_state=RANDOM_STATE).split(df_raw))
print(f"Benchmark: {CV_SPLITS}-fold x {CV_REPEATS} repeats = {len(splits)} paired folds")
bench = {}


def raw_fold(tr, va):
    a, lv = to_matrix(df_raw.iloc[tr])
    b, _ = to_matrix(df_raw.iloc[va], lv)
    return a, b


bench["raw"] = evaluate("Raw corrupted archive", raw_fold, splits, y_all)

if os.path.exists(V2_CLEANED_PATH):
    df_v2 = pd.read_csv(V2_CLEANED_PATH)

    def v2_fold(tr, va):
        a, lv = to_matrix(df_v2.iloc[tr])
        b, _ = to_matrix(df_v2.iloc[va], lv)
        return a, b

    bench["v2_transductive"] = evaluate("v2 cleaned (transductive MICE)", v2_fold, splits, y_all)

_fold_cache = {}


def v3_fold(tr, va, keep_qc):
    key = hash(np.asarray(tr).tobytes())
    if key not in _fold_cache:
        cl = PlanetaryCleaner(fast=True).fit(df_raw.iloc[tr])  # fitted on the training fold only
        _fold_cache[key] = (cl.transform(df_raw.iloc[tr]), cl.transform(df_raw.iloc[va]))
    a, b = _fold_cache[key]
    if not keep_qc:
        a, b = a[ORIGINAL_COLUMNS], b[ORIGINAL_COLUMNS]
    a, lv = to_matrix(a)
    b, _ = to_matrix(b, lv)
    return a, b


t0 = time.time()
bench["v3_core_in_fold"] = evaluate("v3 cleaned, 31 cols (in-fold fit)", lambda tr, va: v3_fold(tr, va, False), splits, y_all)
bench["v3_full_in_fold"] = evaluate("v3 cleaned + qc flags (in-fold fit)", lambda tr, va: v3_fold(tr, va, True), splits, y_all)
print(f"  (v3 in-fold benchmark took {time.time() - t0:.0f}s)")


def paired(a, b, metric):
    d = np.array(bench[b][f"{metric}_folds"]) - np.array(bench[a][f"{metric}_folds"])
    t = stats.ttest_rel(bench[b][f"{metric}_folds"], bench[a][f"{metric}_folds"])
    return {"mean_delta": float(d.mean()), "folds_improved": int((d > 0).sum() if metric == "r2" else (d < 0).sum()),
            "n_folds": int(len(d)), "paired_t_p_value": float(t.pvalue)}


comparisons = {}
for v in [k for k in bench if k != "raw"]:
    comparisons[f"{v}_vs_raw"] = {"r2": paired("raw", v, "r2"), "rmse": paired("raw", v, "rmse")}
if "v2_transductive" in bench:
    comparisons["v3_full_vs_v2"] = {"r2": paired("v2_transductive", "v3_full_in_fold", "r2"),
                                    "rmse": paired("v2_transductive", "v3_full_in_fold", "rmse")}
for k, v in comparisons.items():
    print(f"  {k:<28} ΔR² {v['r2']['mean_delta']:+.4f} (p={v['r2']['paired_t_p_value']:.2g}, "
          f"{v['r2']['folds_improved']}/{v['r2']['n_folds']} folds) | ΔRMSE {v['rmse']['mean_delta']:+.4f}")